# Self-Supervised Learning vs. Finetuning

# 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 2. Define Base Paths

In [ ]:
from pathlib import Path

drive_root = Path('/content/drive/MyDrive')
project_root = drive_root / 'self-supervised-learning'

datasets_dir = project_root / 'datasets'
checkpoints_dir = project_root / 'checkpoints'
logs_dir = project_root / 'logs'
results_dir = project_root / 'results'

for d in [datasets_dir, checkpoints_dir, logs_dir, results_dir]:
    d.mkdir(parents=True, exist_ok=True)

project_root, datasets_dir, checkpoints_dir, logs_dir

# 3. Clone Repo Directly into Google Drive

In [ ]:
import subprocess, os

if not project_root.exists():
    print('Cloning repository into Drive...')
    subprocess.run(['git', 'clone',
                    'https://github.com/sefaburakokcu/self-supervised-learning.git',
                    str(project_root)])
else:
    print('Repository already exists in Drive.')

%cd $project_root

# 4. Install Requirements

In [ ]:
import sys, subprocess
req = project_root / 'requirements.txt'

if req.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req)])
else:
    print('requirements.txt not found.')

# 5. Verify Environment

In [ ]:
import torch, sys

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# 6. List Experiments

In [ ]:
import subprocess

print('Available experiments:')
out = subprocess.run(
    [sys.executable, 'experiments/run_experiment.py', '--list'],
    capture_output=True,
    text=True
)

print(out.stdout)

# 7. Select Experiments to Run

In [ ]:
experiments = [
    'baselines/baseline_resnet18',
    'ssl_pretraining/simclr_resnet18',
    'linear_probing/simclr_resnet18_lp',
]

experiments

# 8. Run Experiments

In [ ]:
import time
from datetime import datetime

results = []

for exp in experiments:
    print('\n' + '='*80)
    print(f'Running: {exp}')
    print('='*80)

    start = time.time()

    ret = subprocess.run([
        sys.executable,
        'experiments/run_experiment.py',
        '--experiment', exp,
        '--device', 'cuda',
        '--checkpoint-dir', str(checkpoints_dir),
        '--log-dir', str(logs_dir),
        '--results-dir', str(results_dir)
    ])

    duration = time.time() - start
    status = 'completed' if ret.returncode == 0 else 'failed'

    results.append({
        'experiment': exp,
        'status': status,
        'duration_sec': duration,
        'timestamp': datetime.now().isoformat()
    })

    print(f'Status: {status}, Duration: {duration:.2f}s')

# 9. Save Summary Metadata

In [ ]:
import json

summary_file = results_dir / 'experiment_summary.json'

with open(summary_file, 'w') as f:
    json.dump(results, f, indent=2)

print('Summary saved to:', summary_file)
results

# 10. List Drive Contents

In [ ]:
for p in project_root.iterdir():
    print(p)